# Sampling Comparison

Ноутбук сравнивает разные solver/method для conditioned sampling в `concat.sampler.Sampler` на одном и том же кейсе,
с одинаковым conditioning и фиксированным seed для честного сравнения.

In [ ]:
import os
import json
import matplotlib.pyplot as plt

from notebook_utils import (
    IMAGE_SIZE,
    build_conditioning,
    denormalize_batch,
    find_latest_run,
    infer_repo_and_data_dirs,
    load_concat_model,
    load_stats,
    load_valid_mask,
    load_validation_case,
    run_concat_sampling_setup,
)
from utils import get_device

REPO_DIR, DATA_ROOT = infer_repo_and_data_dirs()
os.chdir(REPO_DIR)
DEVICE = get_device()

CONCAT_RUN_DIR = None
CHECKPOINT_NAME = 'ema_best_model.pth'
CASE_INDEX = 0
N_TRACKS_RANGE = (1, 4)

SAMPLING_SETUPS = [
    {'name': 'euler_50', 'method': 'euler', 'num_timesteps': 50, 'seed': 2025},
    {'name': 'midpoint_50', 'method': 'midpoint', 'num_timesteps': 50, 'seed': 2025},
    {'name': 'heun3_50', 'method': 'heun3', 'num_timesteps': 50, 'seed': 2025},
    {'name': 'rk4_50', 'method': 'rk4', 'num_timesteps': 50, 'seed': 2025},
    {'name': 'dopri5_50', 'method': 'dopri5', 'num_timesteps': 50, 'seed': 2025, 'rtol': 1e-4, 'atol': 1e-5},
]

if CONCAT_RUN_DIR is None:
    CONCAT_RUN_DIR = find_latest_run(os.path.join(REPO_DIR, 'checkpoints'), CHECKPOINT_NAME)

CHANNEL_MEAN, CHANNEL_STD = load_stats(DATA_ROOT)

print('concat_run =', CONCAT_RUN_DIR)
print('device     =', DEVICE)
print('setups     =')
print(json.dumps(SAMPLING_SETUPS, indent=2))

In [ ]:
model, sampler = load_concat_model(
    run_dir=CONCAT_RUN_DIR,
    image_size=IMAGE_SIZE,
    checkpoint_name=CHECKPOINT_NAME,
    device=DEVICE,
)

valid_mask = load_valid_mask(DATA_ROOT)
clean = load_validation_case(
    data_root=DATA_ROOT,
    channel_mean=CHANNEL_MEAN,
    channel_std=CHANNEL_STD,
    case_index=CASE_INDEX,
    device=DEVICE,
)
mask, observed = build_conditioning(
    clean=clean,
    valid_mask=valid_mask,
    image_size=IMAGE_SIZE,
    n_tracks_range=N_TRACKS_RANGE,
    device=DEVICE,
)

clean_denorm = denormalize_batch(clean.detach().cpu(), CHANNEL_MEAN, CHANNEL_STD)
observed_denorm = denormalize_batch(observed.detach().cpu(), CHANNEL_MEAN, CHANNEL_STD)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(clean_denorm[0, 0], cmap='Blues_r')
axes[0].set_title('Truth concentration')
axes[1].imshow(observed_denorm[0, 0], cmap='Blues_r')
axes[1].set_title('Observed concentration')
axes[2].imshow(mask[0, 0].detach().cpu(), cmap='gray')
axes[2].set_title('Track mask')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

In [ ]:
results = []
for setup in SAMPLING_SETUPS:
    result = run_concat_sampling_setup(
        sampler=sampler,
        mask=mask,
        observed=observed,
        truth=clean,
        size=IMAGE_SIZE,
        config=setup,
        device=DEVICE,
    )
    results.append(result)

summary = [
    {
        'name': result['name'],
        **result['metrics'],
    }
    for result in results
]

summary = sorted(summary, key=lambda row: row['rmse_all'])
print(json.dumps(summary, indent=2))

In [ ]:
n_cols = len(results) + 2
fig, axes = plt.subplots(2, n_cols, figsize=(3.2 * n_cols, 7))
axes[0, 0].imshow(clean_denorm[0, 0], cmap='Blues_r')
axes[0, 0].set_title('Truth concentration')
axes[1, 0].imshow(clean_denorm[0, 1], cmap='viridis')
axes[1, 0].set_title('Truth thickness')
axes[0, 1].imshow(observed_denorm[0, 0], cmap='Blues_r')
axes[0, 1].set_title('Observed concentration')
axes[1, 1].imshow(observed_denorm[0, 1], cmap='viridis')
axes[1, 1].set_title('Observed thickness')

for col, result in enumerate(results, start=2):
    sample_denorm = denormalize_batch(result['sample'], CHANNEL_MEAN, CHANNEL_STD)
    title = f"{result['name']}\nrmse={result['metrics']['rmse_all']:.4f}\nsec={result['metrics']['elapsed_sec']:.2f}"
    axes[0, col].imshow(sample_denorm[0, 0], cmap='Blues_r')
    axes[0, col].set_title(title)
    axes[1, col].imshow(sample_denorm[0, 1], cmap='viridis')
    axes[1, col].set_title(result['name'])

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()

In [ ]:
names = [row['name'] for row in summary]
rmse_all = [row['rmse_all'] for row in summary]
elapsed = [row['elapsed_sec'] for row in summary]
rmse_masked = [row['rmse_masked'] for row in summary]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(names, rmse_all, label='rmse_all')
axes[0].bar(names, rmse_masked, alpha=0.6, label='rmse_masked')
axes[0].set_title('Error by sampling setup')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

axes[1].bar(names, elapsed)
axes[1].set_title('Runtime by sampling setup')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('seconds')

plt.tight_layout()